In [1]:
import os
import json
import time
import random
import asyncio
import aiohttp
import traceback
import textwrap
import ast
from typing import Dict, Any, List
from collections import Counter

from langchain.tools import tool
from bs4 import BeautifulSoup
import agent_config as config

# ============================================
# 辅助函数
# ============================================

def write_file_sync(filepath: str, content: str):
    """同步写入文件 (用于 asyncio.to_thread)"""
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

def read_file_sync(filepath: str) -> str:
    """同步读取文件 (用于 asyncio.to_thread)"""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def get_safe_headers(url: str) -> Dict[str, str]:
    """获取安全的请求头 (包含随机的高质量 Desktop UA)

    逻辑说明:
    1. 随机选择一个主流浏览器的 User-Agent (Chrome, Edge, Firefox, Safari) 以模拟真实用户。
    2. 设置标准的 Accept, Accept-Language 等头部信息。
    3. 特别注意 Accept-Encoding 包含 gzip, deflate 以支持压缩，但不包含 br (Brotli) 以免如果没有相应库导致解压失败。
    """
    pc_user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36 Edg/122.0.0.0",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.3 Safari/605.1.15"
    ]

    return {
        'User-Agent': random.choice(pc_user_agents),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
        'Accept-Encoding': 'gzip, deflate',
        'Referer': url,
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1'
    }



In [2]:
# 1. 强行把当前 Jupyter 内核的工作路径切换到包含 sandbox.py 的这个绝对目录下
%cd /root/ai_projects/fufan/agent_code_files/mycode/langchain1.0/DeepAgent_pachong/

# 2. 验证一下当前目录下是不是真的有 sandbox.py
%ls

/root/ai_projects/fufan/agent_code_files/mycode/langchain1.0/DeepAgent_pachong
6.DeepAgents实现网络爬虫.ipynb  docker_backend.py  sandbox.py         tools.py
agent_config.py                 langgraph.json     save_file/
agent.py                        __pycache__/       spider_demo.py
deep_agents.py                  requirements.txt   spider_workspace/


In [3]:
# docker容器初始化

from sandbox import create_execute_in_sandbox_tool, DockerBackend,initialize_docker_backend
from deepagents.backends import CompositeBackend
from deepagents.backends.filesystem import FilesystemBackend
import docker_backend

# 实例化Docker沙箱执行工具
sandbox_tool = create_execute_in_sandbox_tool(docker_backend)

# 实例化文件系统后端，用于挂载工作目录到容器
fs_backend = FilesystemBackend(root_dir=config.workspace_dir, virtual_mode=True)

# 实现路由，将容器挂载路径与文件系统后端关联
routes = {config.container_mount_path: fs_backend}

# 初始化 Docker 后端，指定容器 ID 并启用文件持久化
docker = initialize_docker_backend(requested_container_id=config.docker_container_id,persist_runtime_files=True)

#  配置 CompositeBackend，将默认 DockerBackend 与文件系统后端合并，确保容器内路径与挂载路径一致
backend = CompositeBackend(default=docker, routes=routes)


2️⃣ 初始化 Docker 沙箱...
⚠️ 无法连接到容器 5236cf3f3150fe26db0ea4d68e8986200f444f4b583aa4adaf6b73fa2a8e021e: Failed to start/attach Docker container: Container 5236cf3f3150fe26db0ea4d68e8986200f444f4b583aa4adaf6b73fa2a8e021e not found.
🔄 正在创建全新的 Docker 容器...
   ✅ Docker 容器已就绪: 94bd968c0053
   💡 提示: 下次运行请更新 agent_config.py 中的 docker_container_id = "94bd968c0053bfbadf12118aabc47a8e6fa431b5b7d31cc78e86ecb0aefd2c27" 以复用此环境
